# 머신러닝을 위한 선형대수학
## 04. 특이값 분해 (Singular Value Decomposition, SVD)

---

### 목차
1. 특이값 분해 예제
2. 특이값 분해 이론
3. SVD의 머신러닝에서의 응용: 영상압축

In [ ]:
# ── 라이브러리 로드 ───

# ── 라이브러리 임포트 ───
# NumPy — 행렬·벡터 연산, 선형대수 핵심 라이브러리
import numpy as np
# Matplotlib — 그래프·벡터 시각화
import matplotlib.pyplot as plt
# patches — 도형·화살표 등 그래픽 요소
import matplotlib.patches as mpatches
# GridSpec — 복잡한 서브플롯 배치
from matplotlib.gridspec import GridSpec

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.unicode_minus'] = False

print('라이브러리 로드 완료')


---

## 1. 특이값 분해 예제

### Positive Semidefinite Matrix (PSDM)

**정의 6.11:** 대칭행렬 $A$ ($A^\top = A$)가 모든 벡터 $x$에 대하여
$$x^\top A x \geq 0$$
이면 $A$를 **양의 준정부호 행렬 (positive semidefinite matrix)** 이라 부른다.

**보기:** 임의의 행렬 $A$에 대하여 $A^\top A$, $AA^\top$는 양의 준정부호 행렬이다.

$$
(1)\; x^\top A^\top A x = (Ax)^\top Ax \geq 0 \qquad
(2)\; x^\top A A^\top x = (A^\top x)^\top A^\top x \geq 0
$$

**정리 6.15:** $n \times n$ 행렬 $A$에 대하여 다음 네 명제는 동치이다.
1. 행렬 $A$가 양의 준정부호 행렬이다.
2. $A$의 모든 eigenvalue는 음의 실수가 아니다.
3. 행렬 $U$가 존재하여 $A = U^\top U$이다.
4. $A$의 모든 sub-determinant는 음의 실수가 아니다.

**정리 6.8 (직교 고유벡터):** $n \times n$ 대칭행렬 $A$에서 $Av_1 = \lambda_1 v_1$, $Av_2 = \lambda_2 v_2$, $\lambda_1 \neq \lambda_2$이면
$$v_1 \cdot v_2 = v_1^\top v_2 = 0$$

**핵심정리 6.9:** $n \times n$ 행렬 $A$가 대칭행렬이면 정규직교행렬 $Q$ ($Q^\top Q = I$)가 존재하여 대각화 가능하다:
$$A = Q^\top DQ, \quad D\text{는 대각행렬}, \quad Q^\top = Q^{-1}$$

### 예제 1: $A = \begin{pmatrix} 1 & 1 & 0 \\ 0 & 0 & 1 \end{pmatrix}$

**Step 1:** $A^\top A$ 계산 및 고유값 구하기

$$
A^\top = \begin{pmatrix}1&0\\1&0\\0&1\end{pmatrix}, \quad
A^\top A = \begin{pmatrix}1&1&0\\1&1&0\\0&0&1\end{pmatrix}
$$

특성다항식:
$$
|\lambda I - A^\top A| = (\lambda-2)(\lambda-1)\lambda = 0 \implies \lambda = 2, 1, 0
$$

**Step 2:** 각 고유값에 대한 고유벡터
- $\lambda=2$: $v_1 = \left(\frac{1}{\sqrt{2}}, \frac{1}{\sqrt{2}}, 0\right)^\top$
- $\lambda=1$: $v_2 = (0, 0, 1)^\top$
- $\lambda=0$: $v_3 = \left(-\frac{1}{\sqrt{2}}, \frac{1}{\sqrt{2}}, 0\right)^\top$

**Step 3:** $u_i = \frac{1}{\sqrt{\lambda_i}} A v_i$ 로 $U$ 구성 ($\lambda > 0$인 경우만)

$$
u_1 = \begin{pmatrix}1\\0\end{pmatrix}, \quad u_2 = \begin{pmatrix}0\\1\end{pmatrix}
\implies U = \begin{pmatrix}1&0\\0&1\end{pmatrix}
$$

$$
\therefore U^\top A V = \begin{pmatrix}\sqrt{2}&0&0\\0&1&0\end{pmatrix}
$$

In [ ]:
# ── 예제 1: A = [[1,1,0],[0,0,1]] SVD 수치 검증 ────────────────────────────
A1 = np.array([[1, 1, 0],
               [0, 0, 1]], dtype=float)

# A^T A 계산
# A^T A — 대칭, 고유값=σ²
AtA = A1.T @ A1
print('A^T A =\n', AtA)

# 고유값·고유벡터 (A^T A는 대칭행렬)
# 고윳값·고유벡터 계산 (Av = λv)
# 대칭행렬 전용 고윳값 분해 (실수 고윳값 보장)
eigvals, eigvecs = np.linalg.eigh(AtA)  # eigh: 대칭행렬 전용
idx = np.argsort(eigvals)[::-1]         # 내림차순 정렬
eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]

print('\nA^T A 고유값 (내림차순):', np.round(eigvals, 6))
print('V (열 = 고유벡터) =\n', np.round(eigvecs, 4))

# numpy SVD
# 특이값 분해 SVD
U, S, Vt = np.linalg.svd(A1, full_matrices=True)
print('\n=== numpy SVD 결과 ===')
print('U =\n', np.round(U, 4))
print('특이값 S =', np.round(S, 4))
print('V^T =\n', np.round(Vt, 4))

# 복원 검증
Lambda = np.zeros((U.shape[0], Vt.shape[0]))
for i, s in enumerate(S):
    Lambda[i, i] = s
A_reconstructed = U @ Lambda @ Vt
print('\nU Λ V^T (복원) =\n', np.round(A_reconstructed, 4))
print('원래 A와 일치:', np.allclose(A1, A_reconstructed))


### 예제 2: $B = \begin{pmatrix}1&1\\1&0\\0&1\end{pmatrix}$

$$
B^\top B = \begin{pmatrix}2&1\\1&2\end{pmatrix},
\quad |\lambda I - B^\top B| = (\lambda-3)(\lambda-1) = 0
\implies \lambda = 3, 1
$$

고유벡터:
$$
v_1 = \begin{pmatrix}\frac{1}{\sqrt{2}}\\\frac{1}{\sqrt{2}}\end{pmatrix},
\quad v_2 = \begin{pmatrix}-\frac{1}{\sqrt{2}}\\\frac{1}{\sqrt{2}}\end{pmatrix}
$$

$u_i = \frac{1}{\sqrt{\lambda_i}} B v_i$ 를 통해 $u_1, u_2$ 계산, $u_3 = u_1 \times u_2$ (벡터곱):

$$
U^\top B V = \begin{pmatrix}\sqrt{3}&0\\0&1\\0&0\end{pmatrix}
$$

In [ ]:
# ── 예제 2: B = [[1,1],[1,0],[0,1]] SVD 수치 검증 ──────────────────────────
B = np.array([[1, 1],
              [1, 0],
              [0, 1]], dtype=float)

# 특이값 분해 SVD
U2, S2, Vt2 = np.linalg.svd(B, full_matrices=True)
print('=== 예제 2 SVD ===')
print('U =\n', np.round(U2, 4))
print('특이값 S =', np.round(S2, 4))  # [√3, 1]
print('V^T =\n', np.round(Vt2, 4))

# Sigma 행렬 구성
# Σ: m×n 대각 (특이값)
Sigma2 = np.zeros((3, 2))
for i, s in enumerate(S2):
    Sigma2[i, i] = s
print('\nΣ (m×n 대각행렬) =\n', np.round(Sigma2, 4))

# U^T B V 검증
result = U2.T @ B @ Vt2.T
print('\nU^T B V =\n', np.round(result, 4))  # 대각행렬 Σ와 일치해야 함


### 예제 3: $A = \begin{pmatrix}4&0\\3&-5\end{pmatrix}$

$$
A^\top A = \begin{pmatrix}4&3\\0&-5\end{pmatrix}\begin{pmatrix}4&0\\3&-5\end{pmatrix}
= \begin{pmatrix}25&-15\\-15&25\end{pmatrix}
$$

$$
\det(\lambda I - A^\top A) = (\lambda-25)^2 - 225 = (\lambda-40)(\lambda-10) = 0
\implies \lambda = 40, 10
$$

특이값: $\sigma_1 = \sqrt{40} = 2\sqrt{10}$, $\sigma_2 = \sqrt{10}$

$$
V = \begin{pmatrix}\frac{1}{\sqrt{2}}&\frac{1}{\sqrt{2}}\\-\frac{1}{\sqrt{2}}&\frac{1}{\sqrt{2}}\end{pmatrix},
\quad
U^\top A V = \begin{pmatrix}\sqrt{40}&0\\0&\sqrt{10}\end{pmatrix}
$$

In [ ]:
# ── 예제 3: A = [[4,0],[3,-5]] SVD ────────────────────────────────────────
A3 = np.array([[4, 0],
               [3, -5]], dtype=float)

# A^T A 고유값
AtA3 = A3.T @ A3
print('A^T A =\n', AtA3)

# 특이값 분해 SVD
U3, S3, Vt3 = np.linalg.svd(A3)
print('\n특이값 S =', np.round(S3, 4))
print('참고: √40 =', round(np.sqrt(40), 4), ', √10 =', round(np.sqrt(10), 4))
print('U =\n', np.round(U3, 4))
print('V^T =\n', np.round(Vt3, 4))

# U^T A V = Σ 검증
Sigma3 = np.diag(S3)
result3 = U3.T @ A3 @ Vt3.T
print('\nU^T A V =\n', np.round(result3, 4))
print('대각행렬 Σ =\n', np.round(Sigma3, 4))
print('일치 여부:', np.allclose(result3, Sigma3))


In [ ]:
# ── 3가지 예제 시각화: 특이값 크기 비교 ─────────────────────────────────────
# Matplotlib — 개념을 그림으로 시각화
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

examples = [
# 특이값 분해 SVD
    ('예제 1\n$A_{2×3}$', np.linalg.svd(A1, compute_uv=False)),
# 특이값 분해 SVD
    ('예제 2\n$B_{3×2}$', np.linalg.svd(B, compute_uv=False)),
# 특이값 분해 SVD
    ('예제 3\n$A_{2×2}$', np.linalg.svd(A3, compute_uv=False)),
]

colors = ['#5C6BC0', '#8E24AA', '#00897B']

for ax, (title, sv), color in zip(axes, examples, colors):
    bars = ax.bar([f'$\\sigma_{i+1}$' for i in range(len(sv))],
                  sv, color=color, alpha=0.85, edgecolor='white', width=0.5)
    for bar, val in zip(bars, sv):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('특이값 (Singular Value)', fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_ylim(0, max(sv) * 1.25)
    ax.grid(True, alpha=0.3, axis='y')

# 전체 figure 제목
plt.suptitle('각 예제의 특이값 크기', fontsize=13, fontweight='bold', y=1.02)
# 서브플롯 간격 자동 조정
plt.tight_layout()
# 그래프 화면 출력
plt.show()


---

## 2. 특이값 분해 이론

### SVD 정의

**정리 7.1:** 행렬의 특이값 분해 (Singular Value Decomposition, SVD)는 $m \times n$ 행렬 $A$를 아래와 같이 분해된 행렬로 표현하는 방법이다.

$$
\boxed{A = U \Lambda V^\top}
$$

| 기호 | 크기 | 성질 |
|------|------|------|
| $U$ | $m \times m$ | 직교행렬: $UU^\top = U^\top U = I_{m \times m}$ |
| $\Lambda$ | $m \times n$ | 대각행렬 |
| $V$ | $n \times n$ | 직교행렬: $VV^\top = V^\top V = I_{n \times n}$ |

$p = \min(m, n)$이라 하면 $\Sigma = \text{diag}(\sigma_1, \sigma_2, \ldots, \sigma_p)$이고
$$\sigma_1 \geq \sigma_2 \geq \cdots \geq \sigma_p \geq 0$$
$\sigma_1, \sigma_2, \ldots, \sigma_p$를 **특이값(singular value)** 이라고 부른다.

### 보조정리

$A$가 $m \times n$ 행렬이라고 하면 $A : R^n \to R^m$인 선형함수이고 $A^\top : R^m \to R^n$인 함수이다. 그러면:
1. $x \in R^n$, $y \in R^m$이면 $\langle Ax, y \rangle = \langle x, A^\top y \rangle$
2. $A(R^n)^\perp = N(A^\top)$
3. $[A^\top(R^m)]^\perp = N(A)$

### SVD 이론 핵심

$A = U\Lambda V^\top$에서:
$$
A^\top A = V\Lambda^\top U^\top U\Lambda V^\top = V\Lambda^\top \Lambda V^\top
$$
따라서 $\Lambda_1 = \Lambda^\top \Lambda = \text{diag}(\sigma_1^2, \ldots, \sigma_n^2)$라 하면
$$(A^\top A)V = V\Lambda_1$$
$\Rightarrow$ **$V$의 열벡터들은 $A^\top A$의 고유벡터, $\sigma_i^2$은 고유값**

마찬가지로 $(AA^\top)U = U\Lambda_2$에서
$\Rightarrow$ **$U$의 열벡터들은 $AA^\top$의 고유벡터**

**정리 7.2:** $A = U\Lambda V^\top$이면 행렬 $V$의 열벡터는 $A^\top A$의 고유벡터이다.

In [ ]:
# ── SVD 구조 시각화 ─────────────────────────────────────────────────────────
np.random.seed(42)
A_demo = np.array([[9.5, 1.4, 7, 0.3],
                   [4,   4,   9, 8  ],
                   [8,   9.1, 6.5, 9.3]])

# 특이값 분해 SVD
U_d, S_d, Vt_d = np.linalg.svd(A_demo, full_matrices=True)
m, n = A_demo.shape
# Λ 행렬 (특이값 대각)
Lambda_d = np.zeros((m, n))
for i, s in enumerate(S_d):
    Lambda_d[i, i] = s

# Matplotlib — 개념을 그림으로 시각화
fig, axes = plt.subplots(1, 4, figsize=(14, 4),
                          gridspec_kw={'width_ratios': [4, 3, 4, 4]})

titles = ['$A$ (3×4)', '$U$ (3×3)', '$\\Lambda$ (3×4)', '$V^\\top$ (4×4)']
mats   = [A_demo, U_d, Lambda_d, Vt_d]
cmaps  = ['Blues', 'Purples', 'Oranges', 'Greens']

for ax, mat, title, cmap in zip(axes, mats, titles, cmaps):
    im = ax.imshow(mat, cmap=cmap, aspect='auto')
    ax.set_title(title, fontsize=12, fontweight='bold')
# 반복: 각 원소/조합에 대해 계산·검증
    for i in range(mat.shape[0]):
# 반복: 각 원소/조합에 대해 계산·검증
        for j in range(mat.shape[1]):
            val = mat[i, j]
            color = 'white' if abs(val) > abs(mat).max() * 0.6 else 'black'
            ax.text(j, i, f'{val:.1f}', ha='center', va='center',
                    fontsize=7, color=color)
    ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(im, ax=ax, shrink=0.8)

# 전체 figure 제목
plt.suptitle('SVD 분해: $A = U\\Lambda V^\\top$', fontsize=13, fontweight='bold', y=1.02)
# 서브플롯 간격 자동 조정
plt.tight_layout()
# 그래프 화면 출력
plt.show()

print('특이값 S =', np.round(S_d, 4))
print('S² (= A^T A 고유값) =', np.round(S_d**2, 4))


In [ ]:
# ── V의 열벡터 = A^T A 고유벡터 검증 ─────────────────────────────────────────
AtA_demo = A_demo.T @ A_demo
# 고윳값·고유벡터 계산 (Av = λv)
# 대칭행렬 전용 고윳값 분해 (실수 고윳값 보장)
eigvals_d, eigvecs_d = np.linalg.eigh(AtA_demo)

# eigh는 오름차순 → 내림차순으로 정렬
idx_d = np.argsort(eigvals_d)[::-1]
eigvals_d = eigvals_d[idx_d]
eigvecs_d = eigvecs_d[:, idx_d]

print('A^T A 고유값  :', np.round(eigvals_d, 4))
print('S² (SVD 특이값²):', np.round(S_d**2, 4))
print('\n일치 여부:', np.allclose(eigvals_d[:len(S_d)], S_d**2, atol=1e-6))

# 고유벡터와 V 비교 (부호 차이 허용)
# SVD의 V 열 = A^TA 고유벡터
V_d = Vt_d.T
print('\nV (SVD) 첫 번째 열:', np.round(V_d[:, 0], 4))
print('eigh 고유벡터 첫 번째 열:', np.round(eigvecs_d[:, 0], 4))
print('(부호만 다를 수 있음 — 방향은 동일)')


---

## 3. SVD의 머신러닝에서의 응용: 영상압축

### Rank-1 행렬과 외적

$rank(A) = 1$인 행렬은 두 벡터의 외적으로 표현 가능하다:
$$A = uv^\top, \quad u \in R^m,\; v \in R^n$$

예: $100 \times 100 = 10000$ 원소를 $2 \times 100 = 200$개 원소로 표현 가능 → **차원 축소**

### SVD를 활용한 압축 원리

**정리 7.4:** $m \times n$ 행렬 $A$의 SVD가 $A = U\Lambda V^\top$이고 $\text{rank}(A) = k$라 하면

$$
A = \sigma_1 u_1 v_1^\top + \sigma_2 u_2 v_2^\top + \cdots + \sigma_k u_k v_k^\top
$$

$\sigma_1 \geq \sigma_2 \geq \cdots \geq \sigma_k$에서 $\sigma_3, \sigma_4$가 $\sigma_1, \sigma_2$에 비해 매우 작다면

$$
A \approx \sigma_1 u_1 v_1^\top + \sigma_2 u_2 v_2^\top
$$

→ 이 원리를 이용하면 **상위 $k$개의 특이값만 사용해 행렬을 근사**할 수 있다.

### 영상 압축 예시

203×233 크기 흑백영상 $A$ ($\text{rank}(A) = 5$):
- 원본 저장 공간: $203 \times 233 = 47299$
- 5개 특이값으로 압축: $5 \times (203 + 233) = 2180$
- **약 21.7배 압축 가능**

In [ ]:
# ── SVD 압축 실습: 행렬을 rank-k 근사로 복원 ──────────────────────────────
A_comp = np.array([[3, 1, 4, 1],
                   [5, 9, 2, 6],
                   [5, 3, 5, 8],
                   [9, 9, 9, 3]], dtype=float)

# 특이값 분해 SVD
U_c, S_c, Vt_c = np.linalg.svd(A_comp)
print('원본 행렬 A =\n', A_comp)
print('\n특이값 S =', np.round(S_c, 4))
print('σ₁ + σ₂가 전체의', round((S_c[0]+S_c[1])/S_c.sum()*100, 1), '% 차지')

# rank-k 근사 함수
# rank-k: 상위 k개 σᵢ uᵢ vᵢ^T 합
# 함수 svd_approx: 알고리즘·목적 설명은 docstring 참고
def svd_approx(A, k):
# 특이값 분해 SVD
    U, S, Vt = np.linalg.svd(A, full_matrices=False)
    return sum(S[i] * np.outer(U[:, i], Vt[i, :]) for i in range(k))

# Matplotlib — 개념을 그림으로 시각화
fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))

vmin, vmax = A_comp.min(), A_comp.max()
im = axes[0].imshow(A_comp, cmap='Blues', vmin=vmin, vmax=vmax)
axes[0].set_title('원본 A', fontsize=11, fontweight='bold')

for ax, k in zip(axes[1:], [1, 2, 3, 4]):
    Ak = svd_approx(A_comp, k)
# Frobenius norm ||A||_F
    err = np.linalg.norm(A_comp - Ak, 'fro')
    ax.imshow(Ak, cmap='Blues', vmin=vmin, vmax=vmax)
    ax.set_title(f'rank-{k} 근사\nerr={err:.2f}', fontsize=10, fontweight='bold')

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])

plt.colorbar(im, ax=axes[-1], shrink=0.8)
# 전체 figure 제목
plt.suptitle('SVD rank-k 근사: $A \\approx \\sum_{i=1}^k \\sigma_i u_i v_i^\\top$',
             fontsize=12, fontweight='bold', y=1.02)
# 서브플롯 간격 자동 조정
plt.tight_layout()
# 그래프 화면 출력
plt.show()


In [ ]:
# ── rank-k 단계별 수치 확인 ──────────────────────────────────────────────────
print('rank-k 근사 결과 비교:')
print(f'{"":6} | {"근사값":^40} | {"Frobenius 오차":^15}')
print('-' * 70)

# 반복: 각 원소/조합에 대해 계산·검증
for k in range(1, 5):
    Ak = svd_approx(A_comp, k)
# Frobenius norm ||A||_F
    err = np.linalg.norm(A_comp - Ak, 'fro')
# Frobenius norm ||A||_F
    ratio = (1 - err / np.linalg.norm(A_comp, 'fro')) * 100
    print(f'rank-{k} | 오차 = {err:.4f} | 원본 보존률 = {ratio:.1f}%')

# σ₃u₃v₃ᵀ + σ₄u₄v₄ᵀ 크기와 σ₁u₁v₁ᵀ 비교
print('\n특이값 상대 크기:')
for i, s in enumerate(S_c):
    print(f'  σ{i+1} = {s:.4f} ({s/S_c[0]*100:.1f}% of σ₁)')


In [ ]:
# ── 실제 이미지 SVD 압축 시뮬레이션 (하트 이미지 재현) ──────────────────────
# 픽셀 아트 하트 생성 (강의 예제 재현)
# 13×13 픽셀 아트 하트
heart = np.zeros((13, 13))
pattern = [
    (1,2),(1,3),(1,9),(1,10),
    (2,1),(2,2),(2,3),(2,4),(2,8),(2,9),(2,10),(2,11),
    (3,1),(3,2),(3,3),(3,4),(3,5),(3,7),(3,8),(3,9),(3,10),(3,11),
    (4,1),(4,2),(4,3),(4,4),(4,5),(4,6),(4,7),(4,8),(4,9),(4,10),(4,11),
    (5,2),(5,3),(5,4),(5,5),(5,6),(5,7),(5,8),(5,9),(5,10),
    (6,3),(6,4),(6,5),(6,6),(6,7),(6,8),(6,9),
    (7,4),(7,5),(7,6),(7,7),(7,8),
    (8,5),(8,6),(8,7),
    (9,6),
]
for r, c in pattern:
    heart[r, c] = 1.0

# 특이값 분해 SVD
U_h, S_h, Vt_h = np.linalg.svd(heart, full_matrices=False)

# Matplotlib — 개념을 그림으로 시각화
fig, axes = plt.subplots(1, 6, figsize=(16, 3.5))
im0 = axes[0].imshow(heart, cmap='gray_r', vmin=0, vmax=1)
# 행렬 rank — 독립 행/열 개수
axes[0].set_title('원본\n(rank=%d)' % np.linalg.matrix_rank(heart),
                   fontsize=10, fontweight='bold')

for ax, k in zip(axes[1:], [1, 2, 3, 4, 5]):
    Hk = svd_approx(heart, k)
    ax.imshow(Hk, cmap='gray_r', vmin=0, vmax=1)
    orig_size = heart.shape[0] * heart.shape[1]
    compressed = k * (heart.shape[0] + heart.shape[1])
    ax.set_title(f'rank-{k}\n{compressed}/{orig_size}', fontsize=10, fontweight='bold')

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])

# 전체 figure 제목
plt.suptitle('SVD 영상 압축: 특이값 수에 따른 화질 변화',
             fontsize=12, fontweight='bold', y=1.05)
# 서브플롯 간격 자동 조정
plt.tight_layout()
# 그래프 화면 출력
plt.show()

print('\n특이값 상위 5개:', np.round(S_h[:5], 4))


In [ ]:
# ── 압축률 vs 화질(PSNR) 분석 ─────────────────────────────────────────────
# Matplotlib — 개념을 그림으로 시각화
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ranks = range(1, min(heart.shape) + 1)
errors = []
compress_ratios = []
orig_size = heart.size

for k in ranks:
    Hk = svd_approx(heart, k)
# Frobenius norm ||A||_F
    err = np.linalg.norm(heart - Hk, 'fro')
    errors.append(err)
    compress_ratios.append(k * sum(heart.shape) / orig_size * 100)

# 누적 특이값 에너지
# 누적 에너지 = Σσᵢ² 비율
cumulative_energy = np.cumsum(S_h**2) / np.sum(S_h**2) * 100

ax1.plot(list(ranks), errors, 'o-', color='#5C6BC0', linewidth=2, markersize=6)
ax1.axhline(y=0.01, color='red', linestyle='--', alpha=0.7, label='오차 < 0.01')
ax1.set_xlabel('사용한 특이값 수 (k)', fontsize=11)
ax1.set_ylabel('Frobenius 오차', fontsize=11)
ax1.set_title('rank-k 근사 오차', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.spines[['top', 'right']].set_visible(False)
ax1.grid(True, alpha=0.3)

ax2.plot(list(ranks), cumulative_energy, 's-', color='#8E24AA', linewidth=2, markersize=6)
ax2.axhline(y=99, color='red', linestyle='--', alpha=0.7, label='99% 에너지')
ax2.set_xlabel('사용한 특이값 수 (k)', fontsize=11)
ax2.set_ylabel('누적 에너지 (%)', fontsize=11)
ax2.set_title('특이값 누적 에너지', fontsize=12, fontweight='bold')
ax2.set_ylim(0, 105)
ax2.legend(fontsize=10)
ax2.spines[['top', 'right']].set_visible(False)
ax2.grid(True, alpha=0.3)

# 전체 figure 제목
plt.suptitle('SVD 압축: 특이값 수에 따른 정보량 분석',
             fontsize=12, fontweight='bold', y=1.02)
# 서브플롯 간격 자동 조정
plt.tight_layout()
# 그래프 화면 출력
plt.show()

# 압축률 정보
# 행렬 rank — 독립 행/열 개수
rank_img = np.linalg.matrix_rank(heart)
print(f'\n이미지 크기: {heart.shape[0]} × {heart.shape[1]} = {orig_size}')
print(f'실제 rank: {rank_img}')
print(f'rank-{rank_img} 압축 시 저장 원소 수: {rank_img * sum(heart.shape)}')
print(f'압축률: {orig_size / (rank_img * sum(heart.shape)):.1f}배')


---

## 핵심 요약

| 개념 | 정의 | 핵심 포인트 |
|------|------|-------------|
| **PSDM** | $x^\top Ax \geq 0$ | $A^\top A$, $AA^\top$ 는 항상 PSDM |
| **SVD** | $A = U\Lambda V^\top$ | 모든 $m \times n$ 행렬에 적용 가능 |
| **특이값** | $\sigma_i = \sqrt{\lambda_i(A^\top A)}$ | $\sigma_1 \geq \sigma_2 \geq \cdots \geq 0$ |
| **V의 열벡터** | $A^\top A$의 고유벡터 | 우 특이벡터(right singular vector) |
| **U의 열벡터** | $AA^\top$의 고유벡터 | 좌 특이벡터(left singular vector) |
| **rank-k 근사** | $A_k = \sum_{i=1}^k \sigma_i u_i v_i^\top$ | 상위 k개 특이값으로 근사 |
| **영상 압축** | $k(m+n) \ll mn$ | 특이값이 빠르게 감소할수록 효율적 |

### 머신러닝과의 연결

- **PCA (주성분 분석)**: 공분산 행렬의 SVD → 분산이 큰 방향으로 차원 축소
- **잠재 의미 분석 (LSA)**: 텍스트 행렬의 SVD → 문서-단어 관계 분석
- **추천 시스템**: 사용자-아이템 행렬의 SVD → 잠재 요인 모델
- **이미지 압축**: 픽셀 행렬의 SVD → 상위 특이값만으로 이미지 복원
- **노이즈 제거**: 작은 특이값은 노이즈에 해당 → 제거하면 신호 개선